<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-10-05-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Practice — 2026-10-05: Statistics — Is That Performance Drop Real, or Just Noise?

**Dataset:** [`diamonds`](https://ggplot2.tidyverse.org/reference/diamonds.html) — 53,940 real diamond listings (carat, cut, color, clarity, physical dimensions, and sale price), fetched at build time from the `seaborn-data` repository's CSV mirror. Real retail data, not a synthetic set.

**The scenario:** You're the QA engineer for a jewelry marketplace's price-prediction model. The model was trained and validated on listings with "colorless" diamonds (color grades D–F). A new sourcing contract means the marketplace is now also receiving a steady stream of lower-color-grade stones (grades G–J) from a different supplier. You pull one week's worth of the new supplier's listings — 500 of them — to check the model before anyone panics or shrugs it off.

Your job isn't to retrain the model. It's to answer the two questions a release manager would ask before touching anything else:

1. Has the **input data** actually shifted in a way that matters, or is this normal sampling noise?
2. Is the model's **prediction error** on the new batch meaningfully worse, or could the apparent gap be explained by chance?

You'll answer both with statistics, not vibes.

**What you should produce:**
1. `train_and_score(...)` — train a `RandomForestRegressor` on the reference (D–F) cohort and return absolute prediction errors on a reference holdout *and* on the new supplier's batch.
2. `bootstrap_ci(...)` — a percentile bootstrap 95% confidence interval for the mean of an array of errors.
3. `permutation_test_diff_means(...)` — a two-sided permutation test on the gap between two cohorts' mean error, returning the observed gap and a p-value.
4. `drift_report(...)` — for each numeric feature, run a Welch's t-test, a Mann–Whitney U test, and a Kolmogorov–Smirnov test comparing the reference and current cohorts, apply a **Bonferroni correction** across all features, and flag which features actually drifted.
5. A 4–6 sentence written verdict: which features drifted (and why that makes physical sense for this dataset), whether the MAE gap is statistically significant, and what 2–3 numbers you'd put on a production monitoring dashboard to catch this earlier next time.

Fill in every function marked `TODO` / `raise NotImplementedError` yourself before you look at the solution below.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

RNG_SEED = 42

DIAMONDS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
NUMERIC_FEATURES = ["carat", "depth", "table", "x", "y", "z"]

In [ ]:
def load_cohorts(current_batch_size: int = 500, seed: int = RNG_SEED):
    """
    Load diamonds.csv and split it into:
      - reference: color grades D, E, F (the "colorless" batch the model
        was trained/validated on)
      - current: a `current_batch_size`-row random sample of color grades
        G, H, I, J (one week of the new supplier's listings)

    TODO:
      1. pd.read_csv(DIAMONDS_URL)
      2. reference = rows where color is in {"D", "E", "F"}
      3. current_full = rows where color is in {"G", "H", "I", "J"}
      4. current = current_full.sample(n=current_batch_size, random_state=seed)
      5. return reference.reset_index(drop=True), current.reset_index(drop=True)
    """
    raise NotImplementedError


reference_df, current_df = load_cohorts()
print(reference_df.shape, current_df.shape)
reference_df.head()

## Part 1 — Train on the reference cohort, measure error on both

In [ ]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    One-hot encode the 'cut' and 'clarity' columns and keep the numeric
    columns in NUMERIC_FEATURES.
    TODO: pd.get_dummies(df[["cut", "clarity"]], drop_first=True), then
    concat with df[NUMERIC_FEATURES].
    """
    raise NotImplementedError


def train_and_score(reference_df: pd.DataFrame, current_df: pd.DataFrame, seed: int = RNG_SEED):
    """
    TODO:
      1. Split reference_df into train/test (70/30, random_state=seed).
      2. Build features with make_features() on the train split; fit a
         RandomForestRegressor(n_estimators=200, max_depth=12, random_state=seed)
         to predict 'price'.
      3. Build features for the reference test split AND for current_df.
         Reindex both to the training columns (fill_value=0) so one-hot
         columns line up even if a category is missing in one cohort.
      4. Predict on both and return (model, ref_errors, cur_errors), where
         each *_errors is an array of |y_true - y_pred|.
    """
    raise NotImplementedError


model, ref_errors, cur_errors = train_and_score(reference_df, current_df)
print("reference MAE:", ref_errors.mean(), "(n=%d)" % len(ref_errors))
print("current MAE:  ", cur_errors.mean(), "(n=%d)" % len(cur_errors))

## Part 2 — Bootstrap confidence intervals & a permutation test

In [ ]:
def bootstrap_ci(values: np.ndarray, n_boot: int = 2000, alpha: float = 0.05, seed: int = RNG_SEED):
    """
    TODO: resample `values` with replacement n_boot times (same size as
    `values` each time), compute the mean of each resample, and return
    (point_estimate, lower, upper) for a (1 - alpha) percentile bootstrap CI
    around the mean. point_estimate is just values.mean().
    """
    raise NotImplementedError


def permutation_test_diff_means(a: np.ndarray, b: np.ndarray, n_perm: int = 5000, seed: int = RNG_SEED):
    """
    TODO: compute observed = b.mean() - a.mean(). Pool a and b, shuffle the
    pooled array n_perm times, split each shuffle back into the first
    len(a) and remaining len(b) values, and recompute the difference each
    time. Return (observed, p_value), where p_value is the two-sided
    fraction of permuted differences at least as extreme (in absolute
    value) as `observed`. Use (count + 1) / (n_perm + 1) so p is never 0.
    """
    raise NotImplementedError


# TODO:
# 1. Compute and print bootstrap_ci(ref_errors) and bootstrap_ci(cur_errors).
# 2. Compute and print permutation_test_diff_means(ref_errors, cur_errors).
# 3. Looking at the two CIs and the p-value: is the MAE gap statistically
#    significant, or could it be explained by sampling noise?

## Part 3 — Did the input features actually drift?

In [ ]:
def drift_report(reference_df: pd.DataFrame, current_df: pd.DataFrame, features: list, alpha: float = 0.05) -> pd.DataFrame:
    """
    For each feature in `features`, compare reference_df[feature] vs.
    current_df[feature] with three two-sample tests:
      - Welch's t-test:      scipy.stats.ttest_ind(a, b, equal_var=False)
      - Mann-Whitney U:      scipy.stats.mannwhitneyu(a, b, alternative="two-sided")
      - Kolmogorov-Smirnov:  scipy.stats.ks_2samp(a, b)

    Apply a Bonferroni correction across len(features) features: a feature
    is flagged as drifted if the SMALLEST of its three raw p-values is
    below alpha / len(features).

    TODO: build and return a DataFrame with one row per feature and columns
    ['feature', 't_p', 'mwu_p', 'ks_p', 'drifted'].
    """
    raise NotImplementedError


drift_table = drift_report(reference_df, current_df, NUMERIC_FEATURES)
drift_table

## Part 4 — Your verdict

In the cell below, write 4–6 sentences answering:
- Which features drifted, and does that make physical sense for *this* dataset (think about what color grade actually correlates with in diamonds)?
- Is the MAE gap between cohorts statistically significant, or within bootstrap/permutation noise?
- What 2–3 numbers would you put on a production monitoring dashboard so this gets caught automatically next time, instead of someone noticing it by hand?

_Your answer here._

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

### Implementation

```python
def load_cohorts(current_batch_size: int = 500, seed: int = RNG_SEED):
    df = pd.read_csv(DIAMONDS_URL)
    reference = df[df["color"].isin(["D", "E", "F"])]
    current_full = df[df["color"].isin(["G", "H", "I", "J"])]
    current = current_full.sample(n=current_batch_size, random_state=seed)
    return reference.reset_index(drop=True), current.reset_index(drop=True)


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    dummies = pd.get_dummies(df[["cut", "clarity"]], drop_first=True)
    return pd.concat([df[NUMERIC_FEATURES].reset_index(drop=True),
                       dummies.reset_index(drop=True)], axis=1)


def train_and_score(reference_df, current_df, seed=RNG_SEED):
    ref_train, ref_test = train_test_split(reference_df, test_size=0.3, random_state=seed)

    X_train = make_features(ref_train)
    y_train = ref_train["price"].to_numpy()
    model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=seed, n_jobs=-1)
    model.fit(X_train, y_train)

    X_ref_test = make_features(ref_test).reindex(columns=X_train.columns, fill_value=0)
    ref_errors = np.abs(ref_test["price"].to_numpy() - model.predict(X_ref_test))

    X_cur = make_features(current_df).reindex(columns=X_train.columns, fill_value=0)
    cur_errors = np.abs(current_df["price"].to_numpy() - model.predict(X_cur))

    return model, ref_errors, cur_errors


def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean() for _ in range(n_boot)])
    point = values.mean()
    lower, upper = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper


def permutation_test_diff_means(a, b, n_perm=5000, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    observed = b.mean() - a.mean()
    pooled = np.concatenate([a, b])
    na = len(a)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        perm_a, perm_b = pooled[:na], pooled[na:]
        if abs(perm_b.mean() - perm_a.mean()) >= abs(observed):
            count += 1
    p_value = (count + 1) / (n_perm + 1)
    return observed, p_value


def drift_report(reference_df, current_df, features, alpha=0.05):
    rows = []
    correction = alpha / len(features)
    for feat in features:
        a, b = reference_df[feat].to_numpy(), current_df[feat].to_numpy()
        t_p = stats.ttest_ind(a, b, equal_var=False).pvalue
        mwu_p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        ks_p = stats.ks_2samp(a, b).pvalue
        drifted = min(t_p, mwu_p, ks_p) < correction
        rows.append({"feature": feat, "t_p": t_p, "mwu_p": mwu_p, "ks_p": ks_p, "drifted": drifted})
    return pd.DataFrame(rows)
```

### Putting it together

```python
model, ref_errors, cur_errors = train_and_score(reference_df, current_df)
print("reference MAE:", ref_errors.mean(), "(n=%d)" % len(ref_errors))
print("current MAE:  ", cur_errors.mean(), "(n=%d)" % len(cur_errors))

print("reference 95% CI:", bootstrap_ci(ref_errors))
print("current 95% CI:  ", bootstrap_ci(cur_errors))

diff, p_value = permutation_test_diff_means(ref_errors, cur_errors)
print(f"MAE gap (current - reference): {diff:.2f}, p-value: {p_value:.4f}")

drift_table = drift_report(reference_df, current_df, NUMERIC_FEATURES)
print(drift_table)
```

With `RNG_SEED = 42` and a 500-row current batch, this prints something very close to:

```
reference MAE: 270.86 (n=7835)
current MAE:   1134.47 (n=500)
reference 95% CI: (270.86, 260.74, 281.19)
current 95% CI:   (1134.47, 1009.07, 1273.23)
MAE gap (current - reference): 863.61, p-value: 0.0002

  feature          t_p         mwu_p          ks_p  drifted
0   carat 5.558286e-17  4.208439e-17  1.757414e-20      True
1   depth 9.068169e-01  7.028594e-01  7.713884e-01     False
2   table 1.104830e-01  1.815970e-01  1.341592e-01     False
3       x 4.956973e-16  1.417296e-16  1.194722e-19      True
4       y 9.214240e-16  2.327849e-16  6.651812e-18      True
5       z 8.073682e-16  7.189342e-17  1.567300e-19      True
```

### Verdict

**`carat`, `x`, `y`, and `z` drifted (all comfortably below the Bonferroni-corrected threshold of 0.05/6 ≈ 0.0083); `depth` and `table` did not.** That's physically sensible: `depth` and `table` are *proportions* (how the stone is cut relative to its own size), which don't depend on color grade, while `carat`, `x`, `y`, `z` are *size* measurements — and lower color-grade stones (G–J) are disproportionately sold at larger carat weights in this market, because buyers trade off color for size at a lower price point. The drift isn't noise; it's a real, explainable shift in what's coming through the door.

**The MAE gap is statistically significant, not noise.** The two 95% CIs (reference ≈ [261, 281], current ≈ [1009, 1273]) don't overlap at all, and the permutation test gives p ≈ 0.0002 — far below any reasonable alpha. A model trained on mostly-small, colorless stones is making roughly 4x larger errors on this new batch, and that gap would show up again and again if you redrew the sample.

**What to put on a dashboard:** (1) a rolling 95% bootstrap CI on MAE per incoming batch, flagged red when it stops overlapping the validation-set CI; (2) per-feature KS-statistic (not just the p-value, which shrinks automatically as batch size grows) for `carat`/`x`/`y`/`z`, since that's the actual magnitude of distributional shift; (3) the fraction of listings outside the training cohort's color range, since that's the leading indicator that would have caught this *before* the error metric moved.

</details>